# 4D-STEM Data → RDF (Radial Distribution Function) 변환

GUI 없이 Jupyter Notebook으로 진행하는 워크플로우

| Step | 내용 |
|------|------|
| 1 | 4D-STEM 데이터 로드 |
| 2 | Center 찾기 & Beam Stopper 검출 |
| 3 | Single Profile 확인 & RDF 파라미터 설정 |
| 4 | Profile Cube 계산 (3D) |
| 5 | Single RDF 계산 & 4-panel plot |
| 6 | RDF Map 계산 (전체 scanning position) |
| 7 | 결과 저장 |

## Step 0: Import

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import time
import os

import RDF_Package
import RDF_preparation

%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

In [ ]:
# RdfParaData: RDF 계산에 필요한 파라미터를 담는 클래스
class RdfParaData:
    def __init__(self):
        self.element = [26, 5, 14]           # 원소 번호 (예: Fe=26, B=5, Si=14)
        self.composition = [0.8, 0.2, 0.1]   # 조성비
        self.phi_fit_begin = 1200             # Background fitting 시작 pixel
        self.phi_fit_end = 1500               # Background fitting 끝 pixel
        self.pixel_begin = 60                 # RDF 계산 시작 pixel
        self.pixel_end = 1600                 # RDF 계산 끝 pixel
        self.pixel_adjust_begin = 0
        self.pixel_adjust_end = 0
        self.pixel_adjust = 0
        self.smooth_en = 1                    # Smoothing 활성화 (1=On, 0=Off)
        self.smooth_range = 0.25              # Smoothing 범위 (fraction)
        self.smooth_strength = 3              # Smoothing 강도
        self.polyfitN_en = 1
        self.polyfitN = 5                     # Polynomial fitting 차수
        self.califactor = 0.0021745           # Camera calibration (1/Angstrom per pixel)
        self.damping_en = 0                   # Damping 활성화 (1=On, 0=Off)
        self.damping_strength = 0
        self.damping_start_point = 10000
        self.autodark_en = 1                  # Auto dark correction
        self.aver_inten = None                # 1D intensity profile (나중에 설정)
        self.L = 10                           # r축 길이 (Angstrom)
        self.rn = 1000                        # r축 포인트 수

## Step 1: 4D-STEM 데이터 로드

데이터 파일 경로와 scan size(x, y)를 입력하세요.

지원 파일 형식: `.npy`, `.mib`, `.raw` (EMPAD), `.tif`, `.h5`, `.hspy`

In [ ]:
#=== 사용자 설정 ===
data_path = "your_data.npy"   # 데이터 파일 경로
scan_x = 64                    # Scanning size X (가로)
scan_y = 64                    # Scanning size Y (세로)

#=== 데이터 로드 ===
ext = os.path.splitext(data_path)[1].lower()

if ext == '.npy':
    data_4d = np.load(data_path, mmap_mode='r')
elif ext == '.mib':
    import read_mib
    mib = read_mib.MIBFile(data_path)
    data_4d = np.array(mib._frames(scan_x * scan_y, 0))
elif ext == '.raw':
    from read_empad import read_empad
    data_4d = read_empad(data_path, EMPAD_shape=(scan_x, scan_y, 130, 128))
elif ext == '.tif':
    from PIL import Image
    data_4d = np.array(Image.open(data_path))
elif ext == '.h5':
    import h5py
    data_4d = h5py.File(data_path)['entry']['data']['data']
elif ext in ['.hspy', '.dm3', '.dm4']:
    import hyperspy.api as hs
    data_4d = hs.load(data_path).data
else:
    raise ValueError(f"지원하지 않는 파일 형식: {ext}")

# 3D → 4D reshape (mib 등 3D로 로드된 경우)
if len(data_4d.shape) < 4:
    pic_shape = data_4d.shape[1:]
    data_4d = data_4d[0:scan_x*scan_y].reshape(scan_x, scan_y, pic_shape[0], pic_shape[1])

print(f"Data shape: {data_4d.shape}")
print(f"  Scan: {data_4d.shape[0]} x {data_4d.shape[1]}")
print(f"  Detector: {data_4d.shape[2]} x {data_4d.shape[3]}")

In [ ]:
# 중앙 diffraction pattern 확인
center_pattern = data_4d[data_4d.shape[0]//2, data_4d.shape[1]//2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# 왼쪽: diffraction pattern (log scale)
im1 = axes[0].imshow(np.log1p(np.abs(center_pattern)), cmap='viridis')
axes[0].set_title('Center Diffraction Pattern (log scale)')
fig.colorbar(im1, ax=axes[0], shrink=0.8)

# 오른쪽: scan map (전체 intensity 합)
scan_map = np.sum(data_4d, axis=(2, 3))
im2 = axes[1].imshow(scan_map, cmap='viridis')
axes[1].set_title('Scan Map (total intensity)')
fig.colorbar(im2, ax=axes[1], shrink=0.8)
plt.tight_layout()
plt.show()

## Step 2: Center 찾기 & Beam Stopper 검출

In [ ]:
# Auto center finding (Hough transform)
try:
    cx, cy, radius = RDF_preparation.test_hough_2(center_pattern)
    print(f"Auto-detected center: ({cx}, {cy}), radius: {radius}")
except Exception as e:
    print(f"Auto detection failed: {e}")
    # 수동으로 설정
    cx, cy, radius = 64, 64, 10

#=== 수동 보정이 필요하면 아래 값을 직접 수정하세요 ===
# cx = 64      # center x
# cy = 64      # center y
# radius = 10  # beam stopper radius

# center 좌표: RDF_preparation 함수에서 사용하는 형식 (1-indexed)
center = [cy + 1, cx + 1]

print(f"사용할 Center: ({cx}, {cy})")
print(f"사용할 Radius: {radius}")

In [ ]:
# Beam stopper 검출
beam_stopper_low = 0.004   # 하한 threshold (max intensity 대비 비율)
beam_stopper_high = 0.050  # 상한 threshold

center_fig = np.array(center_pattern, dtype=float)
try:
    _, beam_stopper = RDF_preparation.auto_beam_stopper(
        center_fig, [cy, cx], beam_stopper_low, beam_stopper_high
    )
    print("Beam stopper 검출 완료")
except Exception as e:
    print(f"Beam stopper 자동 검출 실패: {e}")
    beam_stopper = None

# 결과 확인
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].imshow(np.log1p(np.abs(center_pattern)), cmap='viridis')
axes[0].plot(cx, cy, 'r+', markersize=15, markeredgewidth=2)
circle = plt.Circle((cx, cy), radius, fill=False, color='r', linewidth=1.5)
axes[0].add_patch(circle)
axes[0].set_title('Center Position & Radius')

if beam_stopper is not None:
    axes[1].imshow(beam_stopper, cmap='gray')
    axes[1].set_title('Beam Stopper Mask\n(1=valid, NaN=stopper)')
    
    masked = np.multiply(center_fig, beam_stopper)
    axes[2].imshow(np.log1p(np.abs(masked)), cmap='viridis')
    axes[2].set_title('After Beam Stopper Applied')
else:
    axes[1].text(0.5, 0.5, 'No beam stopper', ha='center', va='center', transform=axes[1].transAxes)
    axes[2].text(0.5, 0.5, 'No beam stopper', ha='center', va='center', transform=axes[2].transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# (선택) Center map 계산: scanning position마다 center를 따로 계산
# 대용량 데이터에서는 시간이 오래 걸릴 수 있음
use_center_map = False   # True로 바꾸면 center map 계산

if use_center_map:
    print("Center map 계산 중...")
    center_map, _ = RDF_preparation.calc_center_map(data_4d)
    print(f"Center map shape: {center_map.shape}")
else:
    center_map = None
    print("단일 center 사용")

## Step 3: Single Profile 확인 & RDF 파라미터 설정

중앙 위치의 diffraction pattern에서 1D radial profile을 계산해서 RDF 파라미터가 적절한지 확인합니다.

In [ ]:
# 중앙 패턴에서 single radial profile 계산
single_pattern = np.array(center_pattern, dtype=float)
if beam_stopper is not None:
    single_pattern = np.multiply(single_pattern, beam_stopper)

profile = RDF_preparation.intensity_average(single_pattern, center)
profile[0:int(radius)] = 0  # beam stopper 영역 제거
profile[profile == 0] = np.nan

plt.figure(figsize=(12, 4))
plt.plot(profile)
plt.title('Single Radial Profile (center pattern)')
plt.xlabel('Pixel')
plt.ylabel('Intensity')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Profile length: {len(profile)} pixels")

In [ ]:
#=== RDF 파라미터 설정 ===
# 위 profile plot을 보고 적절한 값으로 수정하세요.
# pixel_begin/end, phi_fit_begin/end는 profile 길이보다 작아야 합니다.

rdf_para = RdfParaData()

# 원소 & 조성 설정
rdf_para.element = [26, 5, 14]         # 원소 번호 (Fe=26, B=5, Si=14 등)
rdf_para.composition = [0.8, 0.2, 0.1] # 조성비

# Pixel 범위 (profile plot 보고 조정)
rdf_para.pixel_begin = 60               # 시작 pixel
rdf_para.pixel_end = min(1600, len(profile))  # 끝 pixel (profile 길이 이내)
rdf_para.pixel_adjust = 0

# Background fitting 범위
rdf_para.phi_fit_begin = min(1200, rdf_para.pixel_end - 10)
rdf_para.phi_fit_end = min(1500, rdf_para.pixel_end)

# Smoothing
rdf_para.smooth_en = 1           # 0=Off, 1=On
rdf_para.smooth_range = 0.25
rdf_para.smooth_strength = 3

# Polynomial fitting
rdf_para.polyfitN = 5

# Camera calibration
rdf_para.califactor = 0.0021745  # Angstrom^-1 per pixel

# Damping
rdf_para.damping_en = 0          # 0=Off, 1=On
rdf_para.damping_strength = 0
rdf_para.damping_start_point = 10000

# RDF output
rdf_para.L = 10                  # r축 최대값 (Angstrom)
rdf_para.rn = 1000               # r축 포인트 수

rdf_para.autodark_en = 1

print("RDF 파라미터 설정 완료")

## Step 4: Profile Cube 계산 (3D)

모든 scanning position에서 radial profile을 계산하여 3D profile cube를 만듭니다.

`process_area`를 설정하면 일부 영역만 계산할 수 있습니다.

In [ ]:
#=== 처리 영역 설정 (None이면 전체 영역) ===
# 일부 영역만 처리하려면: process_area = [[x1, y1], [x2, y2]]
process_area = None

# 처리 범위 결정
x_start, y_start = 0, 0
x_end, y_end = data_4d.shape[0], data_4d.shape[1]

if process_area is not None:
    x_start = process_area[0][1]
    x_end = process_area[1][1] + 1
    y_start = process_area[0][0]
    y_end = process_area[1][0] + 1

# Profile 최대 길이 = detector 대각선의 절반
profile_length = int(((data_4d.shape[2]/2)**2 + (data_4d.shape[3]/2)**2)**0.5)

print(f"처리 영역: x=[{x_start}:{x_end}], y=[{y_start}:{y_end}]")
print(f"Profile length: {profile_length}")
print(f"총 {(x_end-x_start) * (y_end-y_start)}개 패턴 처리 예정")

#=== Profile cube 계산 ===
profile_map = np.zeros((x_end - x_start, y_end - y_start, profile_length))
t_start = time.time()

for i in np.arange(x_start, x_end):
    for j in np.arange(y_start, y_end):
        single = np.array(data_4d[i, j, :, :], dtype=float)
        
        # Beam stopper 적용
        if beam_stopper is not None and beam_stopper.shape == single.shape:
            single = np.multiply(single, beam_stopper)
        
        # Center map 적용 (있는 경우)
        if center_map is not None:
            c = [center_map[0][i, j] + 1, center_map[1][i, j] + 1]
        else:
            c = center
        
        p = RDF_preparation.intensity_average(single, c)
        p[0:int(radius)] = 0
        
        if p.shape[0] == profile_length:
            profile_map[i - x_start, j - y_start, :] = p
        elif p.shape[0] < profile_length:
            profile_map[i - x_start, j - y_start, :p.shape[0]] = p
        else:
            profile_map[i - x_start, j - y_start, :] = p[:profile_length]
    
    # 진행률 표시
    done = i - x_start + 1
    total = x_end - x_start
    elapsed = time.time() - t_start
    if done > 0:
        eta = elapsed / done * (total - done)
        print(f"\r  Row {done}/{total} ({done/total*100:.0f}%) - "
              f"경과: {elapsed:.1f}s, 남은시간: {eta:.1f}s", end='')

print(f"\n\nProfile cube 계산 완료! Shape: {profile_map.shape}")
print(f"총 소요시간: {time.time() - t_start:.1f}초")

In [ ]:
# Profile cube 확인: 평균 profile & intensity map
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 평균 profile
avg_profile = np.nanmean(profile_map, axis=(0, 1))
axes[0].plot(avg_profile)
axes[0].set_title('Average Radial Profile')
axes[0].set_xlabel('Pixel')
axes[0].set_ylabel('Intensity')
axes[0].grid(True, alpha=0.3)

# 오른쪽: intensity map (profile 전체 합)
temp = copy.deepcopy(profile_map)
temp[np.isnan(temp)] = 0
intensity_map = np.sum(temp, axis=2)
im = axes[1].imshow(intensity_map, cmap='viridis')
axes[1].set_title('Integrated Intensity Map')
fig.colorbar(im, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

## Step 5: Single RDF 계산 & 4-Panel Plot

평균 profile로 RDF를 계산하여 파라미터가 적절한지 확인합니다. 결과가 이상하면 Step 3으로 돌아가 파라미터를 조정하세요.

In [ ]:
# 평균 profile로 single RDF 계산
test_profile = np.nanmean(profile_map, axis=(0, 1)).copy()
test_profile[np.isnan(test_profile)] = 0
test_profile[test_profile == 0] = np.nan

# pixel_end가 profile 길이를 넘지 않도록 보정
rdf_para.pixel_end = min(rdf_para.pixel_end, len(test_profile))
rdf_para.phi_fit_end = min(rdf_para.phi_fit_end, rdf_para.pixel_end)
if rdf_para.phi_fit_begin >= rdf_para.phi_fit_end:
    rdf_para.phi_fit_begin = rdf_para.phi_fit_end - 10

rdf_para.aver_inten = test_profile
rdf_result = RDF_Package.rdf_cal(rdf_para, data_trans=1)

print("RDF 계산 완료!")
print(f"  r range: {rdf_result['r'][0]:.2f} ~ {rdf_result['r'][-1]:.2f} Angstrom")

In [ ]:
# 4-Panel Plot (GUI의 RDF 탭과 동일)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.subplots_adjust(wspace=0.3, hspace=0.35)

background_phi = rdf_result['background_phi']
yfit = rdf_result['yfit']
phi = rdf_result['phi']
parameter_s = rdf_result['parameter_s']
pixel_end = rdf_result['pixel_end']
RIF_pristine = rdf_result['RIF_pristine']
RIF_damped = rdf_result['RIF_damped']
RIF_nosmo = rdf_result['RIF_nosmo']
r = rdf_result['r']
G_corrected = rdf_result['G_corrected']
G_corrected_nod = rdf_result['G_corrected_nod']

# (1) Background Fitting
ax1 = axes[0, 0]
ax1.plot(parameter_s[0:pixel_end], test_profile[0:pixel_end], label='profile')
ax1.plot(parameter_s[0:pixel_end], background_phi[0:pixel_end], label='background')
ax1.set_title('Background Fitting')
ax1.set_xlabel('$\\AA^{-1}$')
ax1.set_ylabel('Intensity')
ax1.legend()
ax1.grid(True, alpha=0.3)

# (2) Polynomial Fitting
ax2 = axes[0, 1]
ax2.plot(phi[0:pixel_end], label='structure factor')
ax2.plot(yfit[0:pixel_end], label='fitting')
ax2.set_title('Polynomial Fitting')
ax2.set_xlabel('pixel')
ax2.set_ylabel('Intensity')
ax2.legend()
ax2.grid(True, alpha=0.3)

# (3) Corrected Structure Factor (RIF)
ax3 = axes[1, 0]
plot_end = np.size(RIF_pristine)
if RIF_nosmo is not None:
    plot_end = min(plot_end, np.size(RIF_nosmo))
if RIF_damped is not None:
    plot_end = min(plot_end, np.size(RIF_damped))

if RIF_nosmo is not None:
    ax3.plot(RIF_nosmo[0:plot_end], label='corrected structure factor')
    if RIF_damped is not None:
        ax3.plot(RIF_damped[0:plot_end], label='smoothed and damped')
    else:
        ax3.plot(RIF_pristine[0:plot_end], label='with smoothing')
else:
    ax3.plot(RIF_pristine[0:plot_end], label='corrected structure factor')
    if RIF_damped is not None:
        ax3.plot(RIF_damped[0:plot_end], label='with damping')
ax3.set_title('Corrected Structure Factor')
ax3.set_xlabel('pixel')
ax3.set_ylabel('Intensity')
ax3.legend()
ax3.grid(True, alpha=0.3)

# (4) RDF
ax4 = axes[1, 1]
if G_corrected_nod is not None:
    ax4.plot(r, G_corrected_nod, label='without damping')
    ax4.plot(r, G_corrected, label='with damping')
    ax4.legend()
else:
    ax4.plot(r, G_corrected, label='RDF')
    ax4.legend()
ax4.set_title('RDF (Radial Distribution Function)')
ax4.set_xlabel('r ($\\AA$)')
ax4.set_ylabel('G(r)')
ax4.grid(True, alpha=0.3)

plt.suptitle('Single RDF Result (Average Profile)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Step 6: RDF Map 계산

모든 scanning position에서 RDF를 계산합니다. Step 5의 결과가 만족스러우면 진행하세요.

In [ ]:
shape_x, shape_y = profile_map.shape[0], profile_map.shape[1]

# 첫 번째 pixel로 출력 크기 결정
temp_para = copy.deepcopy(rdf_para)
temp_para.aver_inten = profile_map[0, 0, :].copy()
temp_para.aver_inten[np.isnan(temp_para.aver_inten)] = 0
temp_para.aver_inten[temp_para.aver_inten == 0] = np.nan
test_result = RDF_Package.rdf_cal(temp_para, data_trans=1)

rdf_test = test_result['G_corrected']
RIF_test = test_result['RIF_damped'] if test_result['RIF_damped'] is not None else test_result['RIF_pristine']
r_axis = test_result['r']

rdf_length = np.size(rdf_test)
rif_length = np.size(RIF_test)

# 출력 배열 준비
rdf_map = np.zeros((shape_x, shape_y, rdf_length))
RIF_map = np.zeros((shape_x, shape_y, rif_length))

print(f"RDF Map 계산 시작: {shape_x} x {shape_y} = {shape_x * shape_y}개 position")
print(f"  RDF length: {rdf_length}, RIF length: {rif_length}")

t_start = time.time()
count = 0
total = shape_x * shape_y

for i in range(shape_x):
    for j in range(shape_y):
        count += 1
        single = profile_map[i, j, :].copy()
        single[np.isnan(single)] = 0
        single[single == 0] = np.nan
        
        temp_para.aver_inten = single
        result = RDF_Package.rdf_cal(temp_para, data_trans=1)
        
        rdf_map[i, j, :] = result['G_corrected']
        if result['RIF_damped'] is not None:
            RIF_map[i, j, :] = result['RIF_damped']
        else:
            RIF_map[i, j, :] = result['RIF_pristine']
    
    # 진행률 표시
    done = i + 1
    elapsed = time.time() - t_start
    eta = elapsed / done * (shape_x - done) if done > 0 else 0
    print(f"\r  Row {done}/{shape_x} ({count}/{total}, {count/total*100:.0f}%) - "
          f"경과: {elapsed:.1f}s, 남은시간: {eta:.1f}s", end='')

print(f"\n\nRDF Map 계산 완료!")
print(f"  RDF map shape: {rdf_map.shape}")
print(f"  RIF map shape: {RIF_map.shape}")
print(f"  총 소요시간: {time.time() - t_start:.1f}초")

In [ ]:
# RDF Map 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: RDF map (r축 전체 평균)
temp_rdf = copy.deepcopy(rdf_map)
temp_rdf[np.isnan(temp_rdf)] = 0
rdf_avg_map = np.mean(temp_rdf, axis=2)
im1 = axes[0].imshow(rdf_avg_map, cmap='viridis')
axes[0].set_title('RDF Map (average over r)')
fig.colorbar(im1, ax=axes[0], shrink=0.8)

# 오른쪽: 전체 평균 RDF
avg_rdf = np.mean(temp_rdf, axis=(0, 1))
axes[1].plot(r_axis, avg_rdf)
axes[1].set_title('Average RDF G(r)')
axes[1].set_xlabel('r ($\\AA$)')
axes[1].set_ylabel('G(r)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 특정 위치의 RDF 확인 (클릭 대신 좌표 직접 지정)
# 여러 위치를 비교할 수 있음
positions = [
    (0, 0),
    (shape_x//2, shape_y//2),
    (shape_x-1, shape_y-1),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 위치 표시
im = axes[0].imshow(rdf_avg_map, cmap='viridis')
colors = ['r', 'g', 'b', 'orange', 'purple']
for idx, (px, py) in enumerate(positions):
    axes[0].plot(py, px, 'o', color=colors[idx % len(colors)], markersize=8,
                 label=f'({px},{py})')
axes[0].legend(loc='upper right')
axes[0].set_title('Selected Positions')
fig.colorbar(im, ax=axes[0], shrink=0.8)

# 오른쪽: 해당 위치들의 RDF
for idx, (px, py) in enumerate(positions):
    axes[1].plot(r_axis, rdf_map[px, py, :], color=colors[idx % len(colors)],
                 label=f'({px},{py})')
axes[1].set_title('RDF at Selected Positions')
axes[1].set_xlabel('r ($\\AA$)')
axes[1].set_ylabel('G(r)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 7: 결과 저장

Profile cube, RDF map, Structure factor map을 저장합니다.

지원 형식: `.npy`, `.mrc`, `.txt`

In [ ]:
#=== 저장 경로 설정 ===
save_dir = "."  # 저장 디렉토리
save_format = "npy"  # "npy", "mrc", "txt" 중 선택

# 파일명 자동 생성
prefix = os.path.splitext(os.path.basename(data_path))[0]

# --- Profile cube 저장 ---
profile_path = os.path.join(save_dir, f"{prefix}_profile_cube.{save_format}")
if save_format == "npy":
    np.save(profile_path, profile_map)
elif save_format == "mrc":
    import mrcfile
    temp = np.transpose(profile_map, axes=(2, 0, 1))[:, ::-1, :]
    with mrcfile.new(profile_path, overwrite=True) as mrc:
        mrc.set_data(temp.astype(np.float32))
elif save_format == "txt":
    np.savetxt(profile_path, profile_map.reshape(-1), delimiter='\n')
print(f"Profile cube 저장: {profile_path}")

# --- RDF map 저장 ---
rdf_map_clean = rdf_map.copy()
rdf_map_clean[np.isnan(rdf_map_clean)] = 0

rdf_path = os.path.join(save_dir, f"{prefix}_rdf_map_{rdf_map.shape[2]}_{rdf_map.shape[0]}_{rdf_map.shape[1]}.{save_format}")
if save_format == "npy":
    np.save(rdf_path, rdf_map_clean)
elif save_format == "mrc":
    import mrcfile
    temp = np.transpose(rdf_map_clean, axes=(2, 0, 1))[:, ::-1, :]
    with mrcfile.new(rdf_path, overwrite=True) as mrc:
        mrc.set_data(temp.astype(np.float32))
elif save_format == "txt":
    np.savetxt(rdf_path, rdf_map_clean.reshape(-1), delimiter=',', fmt='%.18e')
print(f"RDF map 저장: {rdf_path}")

# --- Structure factor (RIF) map 저장 ---
RIF_map_clean = RIF_map.copy()
RIF_map_clean[np.isnan(RIF_map_clean)] = 0

rif_path = os.path.join(save_dir, f"{prefix}_StructureFactor_map.{save_format}")
if save_format == "npy":
    np.save(rif_path, RIF_map_clean)
elif save_format == "mrc":
    import mrcfile
    temp = np.transpose(RIF_map_clean, axes=(2, 0, 1))[:, ::-1, :]
    with mrcfile.new(rif_path, overwrite=True) as mrc:
        mrc.set_data(temp.astype(np.float32))
elif save_format == "txt":
    np.savetxt(rif_path, RIF_map_clean.reshape(-1), delimiter=',', fmt='%.18e')
print(f"Structure factor map 저장: {rif_path}")

# --- r축 데이터 저장 ---
r_path = os.path.join(save_dir, f"{prefix}_r_axis.txt")
np.savetxt(r_path, r_axis, delimiter='\n', fmt='%.18e')
print(f"r축 데이터 저장: {r_path}")

print("\n모든 저장 완료!")

## (Optional) 기존 Profile Cube / RDF Map 로드

이미 저장된 데이터를 불러와서 시각화만 할 수 있습니다.

In [ ]:
# --- 기존 데이터 로드 (필요한 경우만 실행) ---
# Profile cube 로드
# profile_map = np.load("your_data_profile_cube.npy")
# print(f"Profile map loaded: {profile_map.shape}")

# RDF map 로드
# rdf_map = np.load("your_data_rdf_map.npy")
# print(f"RDF map loaded: {rdf_map.shape}")

# r축 로드
# r_axis = np.loadtxt("your_data_r_axis.txt")
# print(f"r axis loaded: {r_axis.shape}")